# 🏥 Painel Executivo de Suporte à Decisão — SESPA / Pará
### Projeto Integrador: Da Ingestão à Decisão

---

## 🎯 1. A Pergunta de Decisão
> *"Cruzando as bases do **INPE (Focos de Calor)**, **DataSUS (Internações Respiratórias)** e **IBGE (População do Censo 2022)**, identificamos os municípios com alto risco de sobrecarga hospitalar. Recomendamos que a **Secretaria de Estado de Saúde Pública do Pará (SESPA)** envie **Kits Preventivos de Insumos Respiratórios** nos próximos **30 dias**, priorizando os **municípios com risco $\ge 70\%*. Se agir, o ganho esperado é a **garantia de atendimento de emergência a municípios críticos**; se errarmos, o custo de um falso positivo é de **R$ 3.500,00** em logística estocável, versus o risco inaceitável de desabastecimento de **R$ 45.000,00** por caso negligenciado."*

* **Decisor:** Secretaria de Estado de Saúde Pública do Pará (SESPA)
* **Ação Concreta:** Distribuição antecipada de cilindros de oxigênio e broncodilatadores nos hospitais regionais de referência
* **Camada Consumida:** Camada Gold ()


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Carregar dados da Camada Gold
df_gold_ind = pd.read_parquet("../data/gold/gold_painel_indicadores_pa.parquet")
df_decisao = pd.read_parquet("../data/gold/gold_recomendacao_decisao_sespa.parquet")

print(f"Painel Gold de Indicadores: {df_gold_ind.shape[0]} registros")
print(f"Tabela de Decisão SESPA: {df_decisao.shape[0]} registros no período operacional")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

## 📈 2. Análise da Dinâmica: Queimadas vs. Internações no Pará
Abaixo avaliamos o comportamento temporal agregado no estado ao longo dos 12 meses de 2023, demonstrando a correlação sazonal durante o período de seca amazônica (julho a novembro).

In [ ]:
# Agregação mensal estadual
mensal_pa = df_gold_ind.groupby("ano_mes").agg(
    total_focos=("total_focos_calor", "sum"),
    total_internacoes=("total_internacoes", "sum"),
    taxa_media_100k=("taxa_internacao_100k", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 6))

cor_focos = "#d95f02"
cor_intern = "#1f78b4"

ax1.set_title("Evolução Temporal no Pará (2023): Focos de Queimada (INPE) vs. Internações Respiratórias (DataSUS)", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Mês de Competência", fontweight="bold")
ax1.set_ylabel("Total de Focos de Calor (INPE)", color=cor_focos, fontweight="bold")
bars = ax1.bar(mensal_pa["ano_mes"], mensal_pa["total_focos"], color=cor_focos, alpha=0.6, width=0.5, label="Focos de Queimada")
ax1.tick_params(axis="y", labelcolor=cor_focos)

ax2 = ax1.twinx()
ax2.set_ylabel("Taxa Média de Internações / 100k hab", color=cor_intern, fontweight="bold")
line = ax2.plot(mensal_pa["ano_mes"], mensal_pa["taxa_media_100k"], color=cor_intern, linewidth=3, marker="o", markersize=8, label="Taxa Internações")
ax2.tick_params(axis="y", labelcolor=cor_intern)

# Destacar período de estiagem
ax1.axvspan(6.5, 10.5, color="yellow", alpha=0.15, label="Período de Estiagem (Jul-Nov)")

fig.tight_layout()
plt.show()

## 🛡️ 3. Avaliação Anti-Vazamento e Desempenho do Modelo
Comparativo entre o **Modelo de Baseline Trivial (Regra)** e o **Classificador Calibrado (Random Forest)** avaliado no período de teste estrito (Setembro a Novembro de 2023).

In [ ]:
df_metricas = pd.DataFrame({
    "Modelo": ["Baseline Trivial (Regra Simples)", "Random Forest Calibrado (SESPA)"],
    "Recall (Sensibilidade)": ["88.29%", "80.18%"],
    "Precisão": ["24.32%", "78.76%"],
    "F2-Score (Foco em Recall)": ["57.85%", "79.89%"]
})

display(df_metricas)

print("-" * 70)
print("Trade-off Econômico-Sanitário da Decisão:")
print("• Custo do Falso Positivo (FP): R$ 3.500,00 (Remanejamento fluvial de insumos estocáveis)")
print("• Custo do Falso Negativo (FN): R$ 45.000,00 (UTI aérea de emergência e risco de vida)")
print("• Razão de Severidade: Um falso negativo é ~12.8 vezes mais custoso que um falso positivo!")

## 📋 4. Tabela de Recomendação e Priorização de Municípios (Novembro/2023)
Abaixo está a **tabela final prescritiva** ordenada por risco, fornecendo ao secretário da SESPA a lista priorizada de municípios para envio imediato de insumos respiratórios.

In [ ]:
# Filtrar para o mês mais recente de decisão (Novembro/2023)
decisao_nov = df_decisao[df_decisao["ano_mes_t0"] == "2023-11"].copy()
ranking_top = decisao_nov.sort_values(by="probabilidade_risco", ascending=False).head(15)

cols_apresentacao = [
    "codigo_ibge", "nome_municipio", "populacao_censo_2022",
    "total_focos_calor", "taxa_internacao_100k", "probabilidade_risco", "decisao_recomendada"
]

display(ranking_top[cols_apresentacao])

In [ ]:
# Visualização gráfica do ranking de risco
plt.figure(figsize=(12, 7))
palette = ["#d95f02" if r >= 0.7 else "#e6ab02" for r in ranking_top["probabilidade_risco"]]

sns.barplot(
    data=ranking_top,
    x="probabilidade_risco",
    y="nome_municipio",
    palette=palette
)

plt.axvline(0.70, color="red", linestyle="--", label="Limiar Alerta Alto (>= 70%)")
plt.axvline(0.40, color="orange", linestyle=":", label="Limiar Alerta Médio (>= 40%)")

plt.title("Top 15 Municípios do Pará Priorizados para Alocação de Kits de Emergência (SESPA)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Probabilidade Estimada de Sobrecarga Respiratória", fontweight="bold")
plt.ylabel("Município", fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 🔍 5. Limitações e Próximos Passos
1. **O que os dados permitem afirmar:** Há forte precedência temporal entre o aumento de queimadas na estiagem e a elevação de internações respiratórias em municípios vulneráveis do Pará, justificando a pré-alocação logística de kits.
2. **Limitações:** A base do SIH/DataSUS reflete internações formais no SUS e não captura atendimentos ambulatoriais leves ou pessoas que não conseguiram chegar a uma unidade de saúde.
3. **Próximos passos recomendados:** Incorporar dados de qualidade do ar (sensores de material particulado {2.5}$) e dados meteorológicos horários de vento do INMET.